# LG전자 스마트팩토리 — AI4I 2020 설비 이상감지·예지보전 8시간 실습

**32장 슬라이드 연계 · Module 1~8 · Google Colab 실행용**

학습 경로: **Data → EDA → Physics → Rules → Machine Learning → Evaluation → Dashboard → Capstone**

> **중요:** UCI AI4I 2020은 **합성된 제조 공정 관측치**입니다. 실제 소형 냉각팬 P04 기능검사 설비에서 수집한 데이터가 아닙니다. `Machine failure`는 해당 행의 고장 *분류 레이블*이며, 미래 고장 시점이나 RUL의 정답은 제공하지 않습니다. 행 순서를 실제 설비의 시간축으로 간주해 조기경보 리드타임이나 MA20 성능을 주장하지 않습니다.

**데이터 출처:** UCI AI4I 2020 Predictive Maintenance Dataset, ID 601, DOI: [10.24432/C5HS5C](https://doi.org/10.24432/C5HS5C), CC BY 4.0. [원본 페이지](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset)

**사용법:** 런타임 → 모두 실행. 데이터 인터넷 연결 오류 발생 시 2번 셀에서 CSV 수동 업로드를 사용합니다. 머신러닝 결과는 실행할 때마다 실데이터로 산출됩니다.

## 준비 · 설치 및 공통 설정

Colab에서는 첫 셀 설치 후 런타임 재시작을 요구할 수 있습니다. 이 경우 다시 순서대로 실행합니다.

In [ ]:
%pip -q install -U ucimlrepo pandas numpy matplotlib plotly scikit-learn joblib

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             precision_score, recall_score, f1_score, average_precision_score,
                             precision_recall_curve, PrecisionRecallDisplay)
from sklearn.inspection import permutation_importance
import joblib

RANDOM_STATE = 42
ROOT = Path('/content/ai4i_colab') if Path('/content').exists() else Path('/mnt/data/ai4i_colab')
ROOT.mkdir(parents=True, exist_ok=True)
print('Output folder:', ROOT)

## Module 1 (Slides 1–4) · 데이터 이해와 데이터 사전

**수행:** ① 원본 10,000행 다운로드 ② 컬럼 확인 ③ 식별자/입력/정답 구분 ④ 고장 유형 분포 확인.

UCI 원본 데이터는 `ucimlrepo.fetch_ucirepo(id=601)`를 사용합니다. **다운로드가 안 되면** UCI 웹사이트에서 `ai4i2020.csv`를 받아 Colab에 업로드하세요.

In [ ]:
CSV = ROOT / 'ai4i2020.csv'
if CSV.exists():
    print('기존 로컬 데이터 사용:', CSV)
    df = pd.read_csv(CSV)
else:
    try:
        # 원본 14개 컬럼을 보존하는 UCI CSV 경로 사용
        import requests, io
        url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv'
        r = requests.get(url, timeout=25)
        r.raise_for_status()
        df = pd.read_csv(io.BytesIO(r.content))
        df.to_csv(CSV, index=False)
        print('UCI 원본 CSV 다운로드 성공')
    except Exception as e:
        print('온라인 데이터 접근 실패:', str(e)[:220])
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError('이 환경에서는 먼저 ai4i2020.csv를 지정된 경로에 저장하세요.') from exc
        print('UCI에서 ai4i2020.csv를 내려받은 후 아래 업로드 창에 제출하세요.')
        uploaded = files.upload()
        selected = next((name for name in uploaded if name.lower().endswith('.csv')), None)
        if selected is None:
            raise RuntimeError('CSV 파일이 업로드되지 않았습니다.')
        df = pd.read_csv(selected)
        df.to_csv(CSV, index=False)

EXPECTED = ['UDI','Product ID','Type','Air temperature [K]','Process temperature [K]',
            'Rotational speed [rpm]','Torque [Nm]','Tool wear [min]','Machine failure',
            'TWF','HDF','PWF','OSF','RNF']
missing = set(EXPECTED) - set(df.columns)
assert not missing, f'필수 컬럼 누락: {missing}'
assert len(df) == 10000, f'행 수 검증 실패: {len(df)}'
df = df[EXPECTED].copy()
print('shape:', df.shape)
display(df.head())

In [ ]:
dictionary = pd.DataFrame([
    ('UDI','식별자','레코드 식별번호','-'),('Product ID','식별자','제품 식별번호','-'),
    ('Type','입력','제품 등급 L/M/H','-'),('Air temperature [K]','입력','공기 온도','K'),
    ('Process temperature [K]','입력','공정 온도','K'),('Rotational speed [rpm]','입력','회전속도','rpm'),
    ('Torque [Nm]','입력','토크','Nm'),('Tool wear [min]','입력','공구 마모 누적시간','min'),
    ('Machine failure','정답','행의 고장 여부','0/1'),
    ('TWF','정답','공구 마모 고장','0/1'),('HDF','정답','열 방출 고장','0/1'),
    ('PWF','정답','동력 고장','0/1'),('OSF','정답','과도 하중 고장','0/1'),
    ('RNF','정답','무작위 고장','0/1')], columns=['컬럼','구분','의미','단위'])
display(dictionary)
dictionary.to_csv(ROOT/'data_dictionary.csv', index=False, encoding='utf-8-sig')
print('고장 유형 플래그 합계 (동시 발생 가능):')
display(df[['TWF','HDF','PWF','OSF','RNF']].sum().to_frame('count').T)
# 원본 데이터에는 Machine failure와 고장 종류 플래그 사이의 소수 불일치가 있을 수 있음
mismatch = ((df[['TWF','HDF','PWF','OSF','RNF']].max(axis=1) != df['Machine failure'])).sum()
print('고장 유형 플래그와 전체 고장 레이블의 불일치 행:', mismatch)

**실습 질문:** `Product ID`와 `UDI`를 모델 입력에 넣지 않는 이유는 무엇인가요? TWF~RNF를 `Machine failure` 예측 입력에 포함시키면 어떤 문제가 생기나요?

## Module 2 (Slides 5–8) · EDA와 데이터 품질

정상·고장 클래스 불균형을 확인하고 변수 분포·상관관계를 분석합니다.

In [ ]:
print('결측치:', int(df.isna().sum().sum()), '중복 UDI:', int(df['UDI'].duplicated().sum()))
print('타깃 분포:')
display(df['Machine failure'].value_counts().rename(index={0:'Normal',1:'Failure'}).to_frame('n'))
print('고장비율:', f"{df['Machine failure'].mean():.2%}")
numcols = ['Air temperature [K]','Process temperature [K]','Rotational speed [rpm]',
           'Torque [Nm]','Tool wear [min]']
display(df[numcols].describe().round(2))
fig,axes=plt.subplots(1,2,figsize=(11,3.5))
df['Machine failure'].value_counts().sort_index().plot.bar(ax=axes[0],title='Normal vs Failure')
axes[0].set_xticklabels(['Normal','Failure'],rotation=0)
axes[1].hist(df['Tool wear [min]'],bins=35,edgecolor='white')
axes[1].set_title('Tool wear distribution')
plt.tight_layout();plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for fail,group in df.groupby('Machine failure'):
    axes[0].scatter(group['Rotational speed [rpm]'],group['Torque [Nm]'],
                    s=7,alpha=.3,label=f'Failure={fail}')
axes[0].set(xlabel='Rotational speed [rpm]',ylabel='Torque [Nm]',title='Speed vs Torque')
axes[0].legend()
axes[1].scatter(df['Air temperature [K]'],df['Process temperature [K]'],s=5,alpha=.25)
axes[1].set(xlabel='Air temp [K]',ylabel='Process temp [K]',title='Temperatures')
plt.tight_layout();plt.show()
display(df[numcols + ['Machine failure']].corr().round(2))

**Module 2 제출:** 고장 비율 1개, 그래프 2개, 중요한 관찰 3문장을 작성해 보세요. **모든 정상으로 예측하는 분류기**의 Accuracy와 Recall을 비교해 보세요.

In [ ]:
baseline = np.zeros(len(df), dtype=int)
y_all = df['Machine failure'].astype(int).to_numpy()
print('Always-normal baseline accuracy:', np.mean(baseline == y_all))
print('Always-normal baseline failure recall:', recall_score(y_all, baseline, zero_division=0))

## Module 3 (Slides 9–12) · 물리 파생변수와 고장 메커니즘

- 온도차: ΔT = T_process − T_air (K)
- 회전 동력: P = τ × 2πN / 60 (W)
- 마모·하중 지표: Tool wear × Torque (min·Nm)

**UCI 합성 데이터 생성 규칙과 실제 설비 안전기준은 다릅니다.**

In [ ]:
df['temp_delta_K'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['power_W'] = df['Torque [Nm]'] * 2 * np.pi * df['Rotational speed [rpm]'] / 60
df['wear_torque'] = df['Tool wear [min]'] * df['Torque [Nm]']
engineered = ['temp_delta_K','power_W','wear_torque']
display(df[engineered+['Machine failure']].describe().round(2))
fig, ax = plt.subplots(1,2,figsize=(12,4))
for val,group in df.groupby('HDF'):
    ax[0].scatter(group['Rotational speed [rpm]'], group['temp_delta_K'],s=6,alpha=.25,label=f'HDF={val}')
ax[0].axvline(1380,ls='--',color='gray');ax[0].axhline(8.6,ls='--',color='gray')
ax[0].set(xlabel='RPM',ylabel='Temp delta [K]',title='Heat dissipation rule')
ax[0].legend()
for val,group in df.groupby('PWF'):
    ax[1].hist(group['power_W'],bins=40,alpha=.55,label=f'PWF={val}')
ax[1].axvline(3500,ls='--',color='gray');ax[1].axvline(9000,ls='--',color='gray')
ax[1].set(xlabel='Power [W]',ylabel='Records',title='Power rule')
ax[1].legend()
plt.tight_layout();plt.show()

## Module 4 (Slides 13–16) · 규칙 기반 탐지와 경고 기준

UCI 원본의 정의 규칙을 재현하는 부분과 **수업용 정상·주의·경고 점수**를 분리합니다. RNF는 관측 변수만으로 결정적으로 찾아낼 수 없습니다. TWF도 일정 마모 구간에서 발생이 확률적으로 할당됩니다.

In [ ]:
# 원본 설명의 규칙을 입력 변수에서 재구성
hdf_rule = (df['temp_delta_K'] < 8.6) & (df['Rotational speed [rpm]'] < 1380)
pwf_rule = (df['power_W'] < 3500) | (df['power_W'] > 9000)
osf_threshold = df['Type'].map({'L':11000,'M':12000,'H':13000})
osf_rule = df['wear_torque'] > osf_threshold
rules = pd.DataFrame({'HDF rule':hdf_rule.astype(int),'PWF rule':pwf_rule.astype(int),
                      'OSF rule':osf_rule.astype(int)},index=df.index)
for rule,label in [('HDF rule','HDF'),('PWF rule','PWF'),('OSF rule','OSF')]:
    print(f'{rule}: mismatches to subtype =',int((rules[rule]!=df[label]).sum()))

# 수업용 주의 점수: 0~1이며 공장 안전기준이 아님
# HDF 예시: RPM이 낮고 열 방출을 위한 온도차가 작을수록 주의
margin_temp = np.clip((10.0 - df['temp_delta_K']) / (10.0 - 8.6),0,1)
margin_rpm  = np.clip((1500.0 - df['Rotational speed [rpm]'])/(1500-1380),0,1)
attention_score = 0.5*margin_temp + 0.5*margin_rpm
status = np.where(hdf_rule,'FAILURE_RULE',np.where(attention_score>=0.8,'WARNING',
                  np.where(attention_score>=0.5,'ATTENTION','NORMAL')))
df['training_risk_score']=attention_score
df['training_status']=status
print('교육용 경고 상태 분포:')
display(pd.Series(status).value_counts().to_frame('count'))
print('원본 HDF와 교육용 WARNING은 다른 개념입니다.')

In [ ]:
rule_any = rules.max(axis=1).to_numpy()
print('세 가지 결정적 규칙의 탐지 결과 (TWF/RNF 미포함):')
print(classification_report(df['Machine failure'],rule_any,zero_division=0))
pd.DataFrame({'actual':df['Machine failure'],'rule_prediction':rule_any,
              'training_status':status,'risk_score':attention_score}).to_csv(
              ROOT/'rule_detector_results.csv',index=False,encoding='utf-8-sig')

**Module 4 질문:** Rule-based 경고가 `FAILURE_RULE`과 다른 이유는 무엇인가요? `Machine failure`가 1인데 3가지 규칙에서 놓친 기록은 무엇과 관련될 수 있나요?

## Module 5 (Slides 17–20) · 머신러닝 기반 고장 분류

**데이터 누출 방지:** `UDI`, `Product ID`, `Machine failure`, TWF~RNF는 모델의 입력에서 제외. 기존 파생변수는 물리 입력으로부터 계산 가능하며 사용합니다.

**Train / Validation / Test = 60 / 20 / 20**, stratified split. 전처리와 모델 학습은 train에만 fit합니다.

In [ ]:
FEATURES=['Type']+numcols+engineered
X=df[FEATURES].copy()
y=df['Machine failure'].astype(int).copy()
X_train, X_hold, y_train, y_hold = train_test_split(
    X,y,test_size=.40,stratify=y,random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(
    X_hold,y_hold,test_size=.5,stratify=y_hold,random_state=RANDOM_STATE)
num_features=[c for c in FEATURES if c!='Type']
prep=ColumnTransformer([
    ('numbers',StandardScaler(),num_features),
    ('category',OneHotEncoder(handle_unknown='ignore'),['Type'])])
models={
    'LogisticRegression': Pipeline([('prep',prep),('model',LogisticRegression(
        max_iter=1500,class_weight='balanced',random_state=RANDOM_STATE))]),
    'RandomForest': Pipeline([('prep',prep),('model',RandomForestClassifier(
        n_estimators=140,max_depth=11,min_samples_leaf=3,class_weight='balanced',
        random_state=RANDOM_STATE,n_jobs=1))])
}
print('Train / Validation / Test:',len(X_train),len(X_val),len(X_test))
for name, m in models.items():
    m.fit(X_train,y_train)
    print('Trained:',name)

In [ ]:
# 검증셋에서만 임계값 최적화; 테스트셋은 선택 이후 1회 사용
threshold_grid=np.linspace(.05,.95,91)
validation_results=[]
selected_thresholds={}
for name,m in models.items():
    prob=m.predict_proba(X_val)[:,1]
    records=[]
    for t in threshold_grid:
        pred=(prob>=t).astype(int)
        records.append((float(t),f1_score(y_val,pred,zero_division=0),
                        recall_score(y_val,pred,zero_division=0),
                        precision_score(y_val,pred,zero_division=0)))
    best=sorted(records,key=lambda z:(-z[1],-z[2],z[0]))[0]
    selected_thresholds[name]=best[0]
    validation_results.append({'model':name,'threshold':best[0],
                               'val_F1':best[1],'val_recall':best[2],
                               'val_precision':best[3],
                               'val_PR_AUC':average_precision_score(y_val,prob)})
validation_table=pd.DataFrame(validation_results).sort_values('val_F1',ascending=False)
display(validation_table.round(4))
chosen_name=validation_table.iloc[0]['model']
chosen_model=models[chosen_name]
chosen_threshold=selected_thresholds[chosen_name]
print('검증셋으로만 선택한 모델 / 임계값:',chosen_name,chosen_threshold)

## Module 6 (Slides 21–24) · 최종 성능 평가 및 설명 가능성

혼동행렬, Precision, Recall, F1, PR-AUC를 **봉인된 테스트 분할**에서 한 번 계산합니다. Feature Importance는 인과적 고장 원인의 증거가 아닙니다.

In [ ]:
test_prob=chosen_model.predict_proba(X_test)[:,1]
test_pred=(test_prob>=chosen_threshold).astype(int)
metrics = {'model':chosen_name,'threshold':float(chosen_threshold),
           'Precision':float(precision_score(y_test,test_pred,zero_division=0)),
           'Recall':float(recall_score(y_test,test_pred,zero_division=0)),
           'F1':float(f1_score(y_test,test_pred,zero_division=0)),
           'PR_AUC':float(average_precision_score(y_test,test_prob)),
           'TP':int(((y_test==1)&(test_pred==1)).sum()),
           'TN':int(((y_test==0)&(test_pred==0)).sum()),
           'FP':int(((y_test==0)&(test_pred==1)).sum()),
           'FN':int(((y_test==1)&(test_pred==0)).sum())}
display(pd.DataFrame([metrics]).round(4))
fig,ax=plt.subplots(1,2,figsize=(11,4))
ConfusionMatrixDisplay.from_predictions(y_test,test_pred,display_labels=['Normal','Failure'],ax=ax[0],cmap='Blues')
PrecisionRecallDisplay.from_predictions(y_test,test_prob,ax=ax[1],name=chosen_name)
plt.tight_layout();plt.show()
(ROOT/'test_metrics.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')

In [ ]:
# permutation_importance: 테스트 분할은 앞에서 1회 성능 보고에 사용하였으므로,
# 설명 예시는 검증셋에 대해 계산합니다.
explain_x=X_val.sample(n=min(300,len(X_val)),random_state=RANDOM_STATE)
explain_y=y_val.loc[explain_x.index]
pi = permutation_importance(chosen_model,explain_x,explain_y,scoring='average_precision',
                            n_repeats=2,random_state=RANDOM_STATE,n_jobs=1)
imp=pd.Series(pi.importances_mean,index=FEATURES).sort_values(ascending=False)
fig,ax=plt.subplots(figsize=(9,4));imp.sort_values().plot.barh(ax=ax)
ax.set_title('Permutation importance (validation, PR-AUC drop)')
plt.tight_layout();plt.show()
display(imp.to_frame('importance').round(5))

## Module 7 (Slides 25–28) · 대시보드와 Human-in-the-loop

**교육용 오프라인 모니터링 대시보드**를 HTML로 저장합니다. UCI에는 동일 설비의 실제 시간 정보가 없으므로 아래 샘플들은 **서로 다른 관측치**로 표시합니다. 실제 센서의 실시간 추세로 해석하면 안 됩니다.

In [ ]:
# 모델이 보지 않은 테스트 관측치에 대한 위험 점수 사용
monitor=X_test.copy()
monitor['actual_failure']=y_test.to_numpy()
monitor['failure_probability']=test_prob
monitor['prediction']=(test_prob>=chosen_threshold).astype(int)
monitor['recommendation']=np.where(monitor['prediction']==1,
    'Operator review / inspection recommended','Continue monitoring')
monitor=monitor.sort_values('failure_probability',ascending=False)
display(monitor.head(12))
fig = px.scatter(monitor.reset_index(drop=True),x='Rotational speed [rpm]',
                 y='Torque [Nm]',color='failure_probability',
                 hover_data=['Type','Tool wear [min]','actual_failure'],
                 title='AI4I Test Records — Predicted Failure Risk (not a time series)',
                 color_continuous_scale='Turbo')
fig.update_layout(width=920,height=520)
fig.show()
html_path=ROOT/'monitoring_dashboard.html'
fig.write_html(str(html_path),include_plotlyjs=True)
print('대시보드 HTML 저장:',html_path)

In [ ]:
# 모의 작업자 판단 로그: 승인 상태는 실습 시 수동 결정하며, 설비를 자동제어하지 않음
review=monitor.head(10).copy().reset_index(drop=True)
review.insert(0,'review_id',[f'REV-{i+1:03d}' for i in range(len(review))])
review['operator_decision']='PENDING_REVIEW'
review['operator_note']=''
cols=['review_id','failure_probability','prediction','recommendation',
      'operator_decision','operator_note']
display(review[cols])
review[cols].to_csv(ROOT/'operator_review_template.csv',index=False,encoding='utf-8-sig')
print('점검 후 담당자가 결정/기록하는 양식을 저장했습니다.')

## Module 8 (Slides 29–32) · 팀별 통합 프로젝트 및 산출물 다운로드

**소형 냉각팬 P01~P06와의 연결:** AI4I는 P04 검사대가 실제로 수집한 기록이 아니라 회전설비 상태 분석을 위한 **가상 교육용 매핑**입니다. 직접적인 실제 효과(정지시간 감소, ROI, 조기 경보 시간)는 이 데이터만으로 입증할 수 없습니다.

팀 발표 질문:
1. 어떤 특성이 고장 레이블과 관련되는가?
2. 물리적 규칙과 머신러닝은 어떤 부분이 다른가?
3. FP/FN은 공장 운영에 어떤 영향을 주는가?
4. 점검 권고 이후 누가 무엇을 판단하는가?
5. 실제 P04 적용을 위해 어떤 추가 현장 정보가 필요한가?

In [ ]:
from zipfile import ZipFile
# 학습 데이터/모델과 평가 산출물 저장. 원본 데이터의 라이선스 표시
joblib.dump(chosen_model, ROOT/'chosen_failure_classifier.joblib')
summary = {
  'dataset':'UCI AI4I 2020, ID 601, CC BY 4.0, DOI 10.24432/C5HS5C',
  'method':'snapshot machine-failure classification, no RUL, no predictive lead time',
  'split':'stratified 60/20/20 (train/validation/test)',
  'selected_model':chosen_name,
  'test_metrics':metrics,
  'limitations':['synthetic dataset','not fan factory sensor records',
                 'no physical asset-specific continuous timestamp',
                 'educational warning thresholds are not safety limits']
}
(ROOT/'experiment_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
zip_path=ROOT/'ai4i_lab_outputs.zip'
with ZipFile(zip_path,'w') as z:
    for f in ROOT.iterdir():
        if f.is_file() and f!=zip_path:
            z.write(f,f.name)
print('산출물 ZIP:',zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Colab이 아니라면 위 경로에서 ZIP 파일을 확인하세요.')

### 평가 루브릭 (100점)

| 항목 | 배점 |
|---|---:|
| 데이터 구조 이해 | 15 |
| EDA 및 시각화 | 15 |
| 고장 메커니즘 분석 | 15 |
| 규칙 기반 이상탐지 | 15 |
| ML 모델 구현 | 15 |
| 오탐·미탐 및 설명 가능성 | 10 |
| 모니터링·조치 권고 | 10 |
| 결과 발표 및 한계 기술 | 5 |

**최종 제출:** `.ipynb` 실행 결과 + `ai4i_lab_outputs.zip` + 5분 발표. 모델 고장 분류를 진정한 미래 고장 예측이나 현장 성능 검증으로 표현하지 않습니다.